# 1. Getting started

In [ ]:
import os
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# 2. Model configuration

In [ ]:
# Model from hugging Face hub
base_model = "meta-llama/Llama-2-7b-hf"

# New paraphrase type dataset
etpc_dataset = "jpwahle/etpc"

# fine-tuned model
new_model = "llama-2-7b-hf-etpc"

# 3. Loading dataset, model, and tokenizer

In [ ]:
dataset = load_dataset("etpc", split="train")

# 4-bit quantizazion configuration

In [ ]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

# 5. Loading Llama 2 model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [ ]:


sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=512,
    output_dir="/tmp",
)



trainer = SFTTrainer(
    model,
    train_dataset=dataset,
    args=sft_config,
)

trainer.train()